# Time-Series Transformer for Energy Forecasting with TensorBoard

**Task:** Predict building electrical power consumption 24 hours ahead (144 steps at 10-minute resolution) using a PyTorch Time-Series Transformer Encoder with self-attention and positional encodings.  
**Data:** `library_featured_v2.csv` (engineered in notebook 05) with 27 temporal, calendar, occupancy, and autoregressive features.  
**Benchmark Target:** Compare attention-based long-range temporal dependencies against TCN, BiLSTM, MLP, and tree models (LightGBM, Extra Trees, XGBoost).

---

### Transformer Architecture for Continuous University BMS Telemetry
While Transformers revolutionized Natural Language Processing, adapting them for continuous numerical energy time-series requires specialized design:
1. **Continuous Feature Projection:** Instead of discrete vocabulary lookups, continuous multivariate feature vectors $\mathbf{x}_t \in \mathbb{R}^{27}$ are projected into a latent representation space:
   $$\mathbf{z}_t = \mathbf{W}_{\text{in}} \mathbf{x}_t + \mathbf{b}_{\text{in}} \in \mathbb{R}^{d_{\text{model}}}$$
2. **Sinusoidal Positional Encoding:** Self-attention is permutation-invariant. To preserve temporal sequencing across the lookback horizon ($L = 36$ steps, representing 6 hours of continuous readings), we inject fixed harmonic positional embeddings:
   $$\begin{aligned}
   PE_{(pos, 2i)} &= \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right) \\
   PE_{(pos, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
   \end{aligned}$$
3. **Multi-Head Self-Attention:** For each attention head, queries ($Q$), keys ($K$), and values ($V$) compute dynamic attention weights across all timesteps:
   $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
   This enables the model to dynamically attend to critical historical moments (e.g. morning lecture start times or yesterday's midday HVAC peak) regardless of temporal distance.
4. **Temporal Pooling & Linear Output:** The sequence representation is pooled across the temporal axis and mapped through a regression head: `Linear(64)` $\to$ `ReLU` $\to$ `Dropout(0.2)` $\to$ `Linear(1)`. The output has **no activation function**, delivering unconstrained physical power forecasts in Watts ($W$).

## 1. Imports & Setup

In [1]:
import os
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure utils path is discoverable
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
if "." not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from utils.tb_logger import TorchTBLogger

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Computation Device: {DEVICE}")

PyTorch Version: 2.14.0+cpu
Computation Device: cpu


## 2. Load Engineered Feature Matrix

In [2]:
DATA_PATH = "library_featured_v2.csv" if os.path.exists("library_featured_v2.csv") else "../library_featured_v2.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded library_featured_v2.csv: 41,248 rows x 37 columns


## 3. Feature Selection & Chronological Train/Val/Test Split

In [3]:
FEATURE_COLUMNS = [
    # Baseline power & occupancy
    "power", "occupancy_count", "occupancy_roll_mean_6", "occupancy_diff_1", "rooms_occupied_est",
    # Cyclical temporal features
    "hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    # BMS operational schedule flags
    "weekend", "is_class_hour", "is_evening_study", "is_night", "is_exam_month", "is_vacation_month",
    # Autoregressive lags & rolling stats
    "power_lag_144", "power_lag_288", "power_lag_1008", "occupancy_lag_144",
    "power_roll_mean_144", "power_roll_std_144", "power_roll_min_144", "power_roll_max_144"
]
TARGET_COLUMN = "target"

# Primary 70 / 30 chronological split
test_split_idx = int(len(df) * 0.70)
train_val_df = df.iloc[:test_split_idx].reset_index(drop=True)
test_df = df.iloc[test_split_idx:].reset_index(drop=True)

# Train / Validation sub-split (85% train / 15% val)
val_split_idx = int(len(train_val_df) * 0.85)
train_df = train_val_df.iloc[:val_split_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_split_idx:].reset_index(drop=True)

print(f"Features: {len(FEATURE_COLUMNS)} variables")
print(f"Train sub-split: {len(train_df):>6,} rows | {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Validation split:{len(val_df):>6,} rows | {val_df['timestamp'].min()} to {val_df['timestamp'].max()}")
print(f"Test split:      {len(test_df):>6,} rows | {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")

Features: 27 variables
Train sub-split: 24,542 rows | 2014-02-26 10:30:00+05:30 to 2016-08-10 12:00:00+05:30
Validation split: 4,331 rows | 2016-08-10 12:10:00+05:30 to 2016-11-20 12:50:00+05:30
Test split:      12,375 rows | 2016-11-20 13:00:00+05:30 to 2017-11-02 23:50:00+05:30


## 4. Normalization & Transformer Sequence Dataset

The Transformer operates on batch inputs of dimension:
$$(B, L, D) = (B, 36, 27)$$

In [4]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_x.fit_transform(train_df[FEATURE_COLUMNS].values)
y_train_scaled = scaler_y.fit_transform(train_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_val_scaled = scaler_x.transform(val_df[FEATURE_COLUMNS].values)
y_val_scaled = scaler_y.transform(val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_x.transform(test_df[FEATURE_COLUMNS].values)
y_test_scaled = scaler_y.transform(test_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

SEQ_LEN = 36

class TransformerSequenceDataset(Dataset):
    def __init__(self, X, y, seq_len=36):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.X) - self.seq_len + 1
        
    def __getitem__(self, idx):
        x_window = self.X[idx : idx + self.seq_len]
        y_target = self.y[idx + self.seq_len - 1]
        return x_window, y_target

BATCH_SIZE = 256

train_dataset = TransformerSequenceDataset(X_train_scaled, y_train_scaled, seq_len=SEQ_LEN)
val_dataset   = TransformerSequenceDataset(X_val_scaled,   y_val_scaled,   seq_len=SEQ_LEN)
test_dataset  = TransformerSequenceDataset(X_test_scaled,  y_test_scaled,  seq_len=SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Transformer Datasets ready (seq_len={SEQ_LEN}):")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Val:   {len(val_dataset):,} samples")
print(f"  Test:  {len(test_dataset):,} samples")

Transformer Datasets ready (seq_len=36):
  Train: 24,507 samples
  Val:   4,296 samples
  Test:  12,340 samples


## 5. PyTorch Time-Series Transformer Implementation

We define:
1. `PositionalEncoding`: Implements sinusoidal harmonic frequency embeddings.
2. `BMS_Transformer`: 
   - Feature Projection: $\mathbb{R}^{27} \to \mathbb{R}^{64}$ (`d_model = 64`).
   - Positional Encoding injection.
   - `nn.TransformerEncoder`: 2 layers, 4 attention heads, `dim_feedforward = 128`, `dropout = 0.1`.
   - Global Temporal Average Pooling across sequence steps.
   - Linear Regression Head: `Linear(64, 32)` $\to$ `ReLU()` $\to$ `Linear(32, 1)` with **no activation**.

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x shape: [B, L, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class BMS_Transformer(nn.Module):
    def __init__(self, in_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=200, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)  # Linear output (no activation)
        )

    def forward(self, x):
        # x: [B, L, D]
        h = self.input_proj(x)
        h = self.pos_encoder(h)
        h_enc = self.transformer_encoder(h)  # [B, L, d_model]
        
        # Temporal mean pooling
        h_pool = h_enc.mean(dim=1)
        out = self.head(h_pool)
        return out.squeeze(-1)

model = BMS_Transformer(
    in_features=len(FEATURE_COLUMNS),
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1
).to(DEVICE)

print(model)
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

BMS_Transformer(
  (input_proj): Linear(in_features=27, out_features=64, bias=True)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()

## 6. Initialize TensorBoard Logger & Computational Graph

In [6]:
log_root = "../runs" if os.path.basename(os.getcwd()) == "notebooks" else "runs"
os.makedirs(log_root, exist_ok=True)

sample_x, _ = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)

tb = TorchTBLogger(
    model_name="Transformer",
    model=model,
    sample_input=sample_x,
    log_root=log_root
)

[TB] 'Transformer' -> ../runs\Transformer
[TB] Launch: tensorboard --logdir=../runs/


[TB] Graph skipped: Tracing failed sanity checks!
ERROR: Graphs differed across invocations!
	Graph diff:
		  graph(%self.1 : __torch__.BMS_Transformer,
		        %x.1 : Tensor):
		    %head : __torch__.torch.nn.modules.container.Sequential = prim::GetAttr[name="head"](%self.1)
		    %transformer_encoder : __torch__.torch.nn.modules.transformer.TransformerEncoder = prim::GetAttr[name="transformer_encoder"](%self.1)
		    %pos_encoder : __torch__.PositionalEncoding = prim::GetAttr[name="pos_encoder"](%self.1)
		    %input_proj : __torch__.torch.nn.modules.linear.Linear = prim::GetAttr[name="input_proj"](%self.1)
		-   %bias.9 : Tensor = prim::GetAttr[name="bias"](%input_proj)
		?         ^
		+   %bias.1 : Tensor = prim::GetAttr[name="bias"](%input_proj)
		?         ^
		-   %weight.9 : Tensor = prim::GetAttr[name="weight"](%input_proj)
		?           ^
		+   %weight.1 : Tensor = prim::GetAttr[name="weight"](%input_proj)
		?           ^
		-   %x : Tensor = aten::linear(%x.1, %weight.9, %bi

## 7. Model Training & Per-Epoch TensorBoard Logging

In [7]:
EPOCHS = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

best_val_loss = float("inf")
checkpoint_path = os.path.join(log_root, "best_Transformer.pt")

print(f"Training Time-Series Transformer on {DEVICE} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        
    avg_train_loss = np.mean(train_losses)
    
    model.eval()
    val_losses = []
    val_preds_list = []
    val_targets_list = []
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            val_losses.append(loss.item())
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(y_b.cpu().numpy())
            
    avg_val_loss = np.mean(val_losses)
    current_lr = scheduler.get_last_lr()[0]
    
    val_preds_W = scaler_y.inverse_transform(np.array(val_preds_list).reshape(-1, 1)).ravel()
    val_targets_W = scaler_y.inverse_transform(np.array(val_targets_list).reshape(-1, 1)).ravel()
    
    val_mae = mean_absolute_error(val_targets_W, val_preds_W)
    val_rmse = np.sqrt(mean_squared_error(val_targets_W, val_preds_W))
    val_r2 = r2_score(val_targets_W, val_preds_W)
    
    tb.log_epoch(
        epoch=epoch,
        train_loss=avg_train_loss,
        val_loss=avg_val_loss,
        y_val=val_targets_W,
        y_pred=val_preds_W,
        lr=current_lr
    )
    
    if epoch % 5 == 0:
        tb.log_gradients(model, epoch)
        
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        
    scheduler.step()
    
    print(f"Epoch {epoch:>2d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val MAE: {val_mae:,.1f} W | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")

print(f"\nTraining complete. Best checkpoint saved to {checkpoint_path}")

Training Time-Series Transformer on cpu for 15 epochs...


Epoch  1/15 | Train Loss: 0.4113 | Val Loss: 0.5128 | Val MAE: 3,611.3 W | Val R²: 0.5547 | LR: 1.00e-03


Epoch  2/15 | Train Loss: 0.2506 | Val Loss: 0.4340 | Val MAE: 3,259.8 W | Val R²: 0.6226 | LR: 9.89e-04


Epoch  3/15 | Train Loss: 0.1997 | Val Loss: 0.4754 | Val MAE: 3,264.4 W | Val R²: 0.5858 | LR: 9.57e-04


Epoch  4/15 | Train Loss: 0.1698 | Val Loss: 0.4723 | Val MAE: 3,269.4 W | Val R²: 0.5877 | LR: 9.05e-04


...

Epoch 11/15 | Train Loss: 0.1008 | Val Loss: 0.5025 | Val MAE: 3,354.0 W | Val R²: 0.5621 | LR: 2.58e-04


Epoch 12/15 | Train Loss: 0.0969 | Val Loss: 0.4849 | Val MAE: 3,279.2 W | Val R²: 0.5772 | LR: 1.74e-04


Epoch 13/15 | Train Loss: 0.0943 | Val Loss: 0.5129 | Val MAE: 3,301.8 W | Val R²: 0.5527 | LR: 1.05e-04


Epoch 14/15 | Train Loss: 0.0930 | Val Loss: 0.5044 | Val MAE: 3,307.9 W | Val R²: 0.5601 | LR: 5.28e-05


Epoch 15/15 | Train Loss: 0.0921 | Val Loss: 0.5022 | Val MAE: 3,314.5 W | Val R²: 0.5621 | LR: 2.08e-05

Training complete. Best checkpoint saved to ../runs\best_Transformer.pt


## 8. Final Test Evaluation & Benchmark Comparison

In [8]:
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

# Predict on test set
test_preds_list = []
with torch.no_grad():
    for x_b, _ in test_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        test_preds_list.extend(preds.cpu().numpy())

test_preds_W = scaler_y.inverse_transform(np.array(test_preds_list).reshape(-1, 1)).ravel()
test_targets_W = test_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

# Predict on train set for gap check
train_val_dataset = TransformerSequenceDataset(
    scaler_x.transform(train_val_df[FEATURE_COLUMNS].values),
    scaler_y.transform(train_val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel(),
    seq_len=SEQ_LEN
)
train_val_loader = DataLoader(train_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_preds_list = []
with torch.no_grad():
    for x_b, _ in train_val_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        train_preds_list.extend(preds.cpu().numpy())

train_preds_W = scaler_y.inverse_transform(np.array(train_preds_list).reshape(-1, 1)).ravel()
train_targets_W = train_val_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

test_mae = mean_absolute_error(test_targets_W, test_preds_W)
test_rmse = np.sqrt(mean_squared_error(test_targets_W, test_preds_W))
test_r2 = r2_score(test_targets_W, test_preds_W)
train_r2 = r2_score(train_targets_W, train_preds_W)
r2_gap = train_r2 - test_r2

print("=" * 55)
print(f"  PyTorch Transformer Final Benchmark Results (Test Set)")
print("=" * 55)
print(f"  Test MAE:   {test_mae:,.1f} W")
print(f"  Test RMSE:  {test_rmse:,.1f} W")
print(f"  Test R²:    {test_r2:.4f}")
print(f"  Train R²:   {train_r2:.4f}")
print(f"  R² Gap:     {r2_gap:.4f}")
print("=" * 55)

  PyTorch Transformer Final Benchmark Results (Test Set)
  Test MAE:   5,029.9 W
  Test RMSE:  6,886.9 W
  Test R²:    0.2511
  Train R²:   0.8506
  R² Gap:     0.5995


## 9. Log Final Metrics, Prediction Plots & HParams to TensorBoard

In [9]:
tb.log_final(
    y_train=train_targets_W,
    y_pred_train=train_preds_W,
    y_test=test_targets_W,
    y_pred_test=test_preds_W,
    hparams={
        "architecture": "Transformer (2-layer, 4-head, d_model=64)",
        "seq_len": SEQ_LEN,
        "d_model": 64,
        "nhead": 4,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_init": 1e-3,
        "dropout": 0.1,
        "optimizer": "AdamW",
        "output_act": "Linear"
    }
)

tb.log_prediction_plots(y_test=test_targets_W, y_pred=test_preds_W, n_plot=500)
tb.log_scatter_plot(y_test=test_targets_W, y_pred=test_preds_W)

tb.close()
print("TensorBoard logging complete. Logs written to:", log_root)


[Transformer] Train  MAE=1783W  R2=0.851
[Transformer] Test   MAE=5030W  R2=0.251


[TB] Writer closed: 'Transformer'
TensorBoard logging complete. Logs written to: ../runs


## 10. Grand Benchmark Comparison: All 7 Models

| Model | Model Family | Test MAE (W) | Test RMSE (W) | Test R² | Train-Test R² Gap | Best Use Case |
|---|---|---|---|---|---|---|
| **LightGBM** | Gradient Boosted Trees | ~3,340 | ~4,580 | ~0.672 | 0.210 | Rapid edge inference |
| **Extra Trees** | Randomized Ensembles | ~3,410 | ~4,690 | ~0.655 | 0.280 | Robust to extreme outliers |
| **XGBoost** | Gradient Boosted Trees | 3,180 | 4,405 | 0.693 | 0.170 | High-accuracy tabular baseline |
| **PyTorch MLP** | Deep Tabular NN | ~3,469 | ~4,636 | ~0.660 | 0.168 | Dense continuous representations |
| **PyTorch BiLSTM** | Recurrent Sequential | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | Autocorrelation & diurnal memory |
| **PyTorch TCN** | Dilated Causal Conv | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | Fast parallel multi-scale receptive field |
| **PyTorch Transformer**| Self-Attention Encoder | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | Global temporal dependency modeling |

### How to View Side-by-Side in TensorBoard
To launch the unified comparison dashboard:
```bash
cd d:\Internship\-library-energy-forecasting
tensorboard --logdir=runs/
```
Open **http://localhost:6006** in your browser:
- **Scalars Tab:** Compare real-time training and validation loss curves across all deep models.
- **Images Tab:** Inspect time-series actual vs predicted overlays and scatter plots for every architecture.
- **HParams Tab:** Filter, sort, and inspect all 7 models side-by-side in one consolidated table for your committee presentation.